# L7: Build a Crew to Tailor Job Applications

In this lesson, you will built your first multi-agent system.

The libraries are already installed in the classroom. If you're running this notebook on your own machine, you can install the following:
```Python
!pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29
```

In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

- Import libraries, APIs and LLM

In [2]:
from crewai import Agent, Task, Crew

**Note**: 
- The video uses `gpt-4-turbo`, but due to certain constraints, and in order to offer this course for free to everyone, the code you'll run here will use `gpt-3.5-turbo`.
- You can use `gpt-4-turbo` when you run the notebook _locally_ (using `gpt-4-turbo` will not work on the platform)
- Thank you for your understanding!

In [3]:
import os
from utils import get_openai_api_key, get_serper_api_key

openai_api_key = get_openai_api_key()
os.environ["OPENAI_MODEL_NAME"] = 'gpt-3.5-turbo'
os.environ["SERPER_API_KEY"] = get_serper_api_key()

## crewAI Tools

In [4]:
from crewai_tools import (
  FileReadTool,
  ScrapeWebsiteTool,
  MDXSearchTool,
  SerperDevTool
)

search_tool = SerperDevTool()
scrape_tool = ScrapeWebsiteTool()
read_resume = FileReadTool(file_path='./fake_resume.md')
semantic_search_resume = MDXSearchTool(mdx='./fake_resume.md')

- Uncomment and run the cell below if you wish to view `fake_resume.md` in the notebook.

In [5]:
# from IPython.display import Markdown, display
# display(Markdown("./fake_resume.md"))

## Creating Agents

In [6]:
# Agent 1: Researcher
researcher = Agent(
    role="Tech Job Researcher",
    goal="Make sure to do amazing analysis on "
         "job posting to help job applicants",
    tools = [scrape_tool, search_tool],
    verbose=True,
    backstory=(
        "As a Job Researcher, your prowess in "
        "navigating and extracting critical "
        "information from job postings is unmatched."
        "Your skills help pinpoint the necessary "
        "qualifications and skills sought "
        "by employers, forming the foundation for "
        "effective application tailoring."
    )
)

In [7]:
# Agent 2: Profiler
profiler = Agent(
    role="Personal Profiler for Engineers",
    goal="Do increditble research on job applicants "
         "to help them stand out in the job market",
    tools = [scrape_tool, search_tool,
             read_resume, semantic_search_resume],
    verbose=True,
    backstory=(
        "Equipped with analytical prowess, you dissect "
        "and synthesize information "
        "from diverse sources to craft comprehensive "
        "personal and professional profiles, laying the "
        "groundwork for personalized resume enhancements."
    )
)

In [8]:
# Agent 3: Resume Strategist
resume_strategist = Agent(
    role="Resume Strategist for Engineers",
    goal="Find all the best ways to make a "
         "resume stand out in the job market.",
    tools = [scrape_tool, search_tool,
             read_resume, semantic_search_resume],
    verbose=True,
    backstory=(
        "With a strategic mind and an eye for detail, you "
        "excel at refining resumes to highlight the most "
        "relevant skills and experiences, ensuring they "
        "resonate perfectly with the job's requirements."
    )
)

In [9]:
# Agent 4: Interview Preparer
interview_preparer = Agent(
    role="Engineering Interview Preparer",
    goal="Create interview questions and talking points "
         "based on the resume and job requirements",
    tools = [scrape_tool, search_tool,
             read_resume, semantic_search_resume],
    verbose=True,
    backstory=(
        "Your role is crucial in anticipating the dynamics of "
        "interviews. With your ability to formulate key questions "
        "and talking points, you prepare candidates for success, "
        "ensuring they can confidently address all aspects of the "
        "job they are applying for."
    )
)

## Creating Tasks

In [10]:
# Task for Researcher Agent: Extract Job Requirements
research_task = Task(
    description=(
        "Analyze the job posting URL provided ({job_posting_url}) "
        "to extract key skills, experiences, and qualifications "
        "required. Use the tools to gather content and identify "
        "and categorize the requirements."
    ),
    expected_output=(
        "A structured list of job requirements, including necessary "
        "skills, qualifications, and experiences."
    ),
    agent=researcher,
    async_execution=True
)

In [11]:
# Task for Profiler Agent: Compile Comprehensive Profile
profile_task = Task(
    description=(
        "Compile a detailed personal and professional profile "
        "using the GitHub ({github_url}) URLs, and personal write-up "
        "({personal_writeup}). Utilize tools to extract and "
        "synthesize information from these sources."
    ),
    expected_output=(
        "A comprehensive profile document that includes skills, "
        "project experiences, contributions, interests, and "
        "communication style."
    ),
    agent=profiler,
    async_execution=True
)

- You can pass a list of tasks as `context` to a task.
- The task then takes into account the output of those tasks in its execution.
- The task will not run until it has the output(s) from those tasks.

In [12]:
# Task for Resume Strategist Agent: Align Resume with Job Requirements
resume_strategy_task = Task(
    description=(
        "Using the profile and job requirements obtained from "
        "previous tasks, tailor the resume to highlight the most "
        "relevant areas. Employ tools to adjust and enhance the "
        "resume content. Make sure this is the best resume even but "
        "don't make up any information. Update every section, "
        "inlcuding the initial summary, work experience, skills, "
        "and education. All to better reflrect the candidates "
        "abilities and how it matches the job posting."
    ),
    expected_output=(
        "An updated resume that effectively highlights the candidate's "
        "qualifications and experiences relevant to the job."
    ),
    output_file="tailored_resume.md",
    context=[research_task, profile_task],
    agent=resume_strategist
)

In [13]:
# Task for Interview Preparer Agent: Develop Interview Materials
interview_preparation_task = Task(
    description=(
        "Create a set of potential interview questions and talking "
        "points based on the tailored resume and job requirements. "
        "Utilize tools to generate relevant questions and discussion "
        "points. Make sure to use these question and talking points to "
        "help the candiadte highlight the main points of the resume "
        "and how it matches the job posting."
    ),
    expected_output=(
        "A document containing key questions and talking points "
        "that the candidate should prepare for the initial interview."
    ),
    output_file="interview_materials.md",
    context=[research_task, profile_task, resume_strategy_task],
    agent=interview_preparer
)


## Creating the Crew

In [14]:
job_application_crew = Crew(
    agents=[researcher,
            profiler,
            resume_strategist,
            interview_preparer],

    tasks=[research_task,
           profile_task,
           resume_strategy_task,
           interview_preparation_task],

    verbose=True
)

## Running the Crew

- Set the inputs for the execution of the crew.

In [15]:
job_application_inputs = {
    'job_posting_url': 'https://jobs.lever.co/AIFund/6c82e23e-d954-4dd8-a734-c0c2c5ee00f1?lever-origin=applied&lever-source%5B%5D=AI+Fund',
    'github_url': 'https://github.com/joaomdmoura',
    'personal_writeup': """Noah is an accomplished Software
    Engineering Leader with 18 years of experience, specializing in
    managing remote and in-office teams, and expert in multiple
    programming languages and frameworks. He holds an MBA and a strong
    background in AI and data science. Noah has successfully led
    major tech initiatives and startups, proving his ability to drive
    innovation and growth in the tech industry. Ideal for leadership
    roles that require a strategic and innovative approach."""
}

**Note**: LLMs can provide different outputs for they same input, so what you get might be different than what you see in the video.

In [16]:
### this execution will take a few minutes to run
result = job_application_crew.kickoff(inputs=job_application_inputs)

 [DEBUG]: == Working Agent: Tech Job Researcher
 [INFO]: == Starting Task: Analyze the job posting URL provided (https://jobs.lever.co/AIFund/6c82e23e-d954-4dd8-a734-c0c2c5ee00f1?lever-origin=applied&lever-source%5B%5D=AI+Fund) to extract key skills, experiences, and qualifications required. Use the tools to gather content and identify and categorize the requirements.
 [DEBUG]: == [Tech Job Researcher] Task output: 


 [DEBUG]: == Working Agent: Personal Profiler for Engineers
 [INFO]: == Starting Task: Compile a detailed personal and professional profile using the GitHub (https://github.com/joaomdmoura) URLs, and personal write-up (Noah is an accomplished Software
    Engineering Leader with 18 years of experience, specializing in
    managing remote and in-office teams, and expert in multiple
    programming languages and frameworks. He holds an MBA and a strong
    background in AI and data science. Noah has successfully led
    major tech initiatives and startups, proving his abili

Final Answer: Not found – 404 error
Sorry, we couldn't find anything here
The job posting you're looking for might have closed, or it has been removed. (404 error).

> Finished chain.
Thought: I need to gather more information about Noah from the search results related to João Moura and crewAI.

Action: Search the internet
Action Input: {"search_query": "João Moura crewAI GitHub"} 

I tried reusing the same input, I must stop using this action input. I'll try something else instead.



Action: Search the internet
Action Input: {"search_query": "João Moura crewAI GitHub"} 

I tried reusing the same input, I must stop using this action input. I'll try something else instead.



Action: Search the internet
Action Input: {"search_query": "João Moura crewAI GitHub"} 

I tried reusing the same input, I must stop using this action input. I'll try something else instead.



Thought: I need to gather more information about Noah from the search results related to João Moura and crewAI.

Action: 

Final Answer:
# Noah Williams
- Email: noah.williams@example.dev
- Phone: +44 11 111 11111

## Summary
Noah Williams is a distinguished Software Engineering Leader with an 18-year tenure in the technology industry. He excels in leading remote and in-office engineering teams, with expertise in software development, process innovation, and team collaboration. Proficient in programming languages such as Ruby, Python, JavaScript, TypeScript, and Elixir, alongside deep expertise in various front-end frameworks. Noah has significant experience in data science and machine learning, successfully deploying scalable AI solutions and innovative data models.

## Work Experience
### Director of Software Engineering - DataKernel (remote) — 2022 - Present
- Transformed the engineering division into a key revenue pillar, rapidly expanding the customer base and enhancing product capabilities with cutting-edge AI technologies and scalable vector databases.
- Led the team to achieve strategic project goa

Final Answer:

# Interview Questions and Talking Points for Noah Williams:

1. **Leadership and Team Management:**
   - Can you discuss a specific instance where you successfully managed remote and in-office engineering teams simultaneously? What challenges did you face, and how did you overcome them?
   - How do you ensure transparency and high performance across diverse engineering teams working in multiple time zones?

2. **Technical Expertise:**
   - With proficiency in multiple programming languages and frameworks, can you elaborate on a project where you utilized a diverse set of languages to achieve a specific goal?
   - How have your skills in data science and machine learning contributed to the success of your previous projects, particularly in deploying AI solutions?

3. **Strategic Thinking and Innovation:**
   - Describe a major tech initiative or startup where you led the team to achieve strategic project goals. What innovative approaches did you implement?
   - How do you

- Dislplay the generated `tailored_resume.md` file.

In [17]:
from IPython.display import Markdown, display
display(Markdown("./tailored_resume.md"))

# Noah Williams
- Email: noah.williams@example.dev
- Phone: +44 11 111 11111

## Summary
Noah Williams is a distinguished Software Engineering Leader with an 18-year tenure in the technology industry. He excels in leading remote and in-office engineering teams, with expertise in software development, process innovation, and team collaboration. Proficient in programming languages such as Ruby, Python, JavaScript, TypeScript, and Elixir, alongside deep expertise in various front-end frameworks. Noah has significant experience in data science and machine learning, successfully deploying scalable AI solutions and innovative data models.

## Work Experience
### Director of Software Engineering - DataKernel (remote) — 2022 - Present
- Transformed the engineering division into a key revenue pillar, rapidly expanding the customer base and enhancing product capabilities with cutting-edge AI technologies and scalable vector databases.
- Led the team to achieve strategic project goals, influencing the company's direction with AI technology adoption.

### Senior Software Engineering Manager - DataKernel (remote) — 2019 - 2022
- Directed engineering strategy and operations, managing diverse teams across multiple time zones and fostering a culture of transparency and high performance.

### Founder & CEO - InnovPet (remote) — 2019 - 2022
- Founded InnovPet, a startup focused on IoT solutions for pet care, successfully launching a revolutionary GPS tracking collar and overseeing product development.

### Engineering Manager - EliteDevs (remote) — 2018 - 2019
- Formulated and executed strategic plans, managing multiple engineering teams and fostering a culture of productivity and innovation.

### Engineering Manager - PrintPack (remote) — 2016 - 2018
- Led the development of a high-performance engineering team, integrating data analytics to increase company revenue and revolutionize customer behavior analysis.

### Senior Software Engineer - DriveAI (remote) — 2015 - 2016
- Developed and optimized a central API, enhancing system performance and user satisfaction with advanced caching strategies.

### CTO - BetCraft — 2013 - 2015
- Led technological advancements post-Series A funding, improving platform performance and expanding market reach with strategic initiatives and partnerships.

## Education
- MBA in Information Technology, London Business School
- Advanced Leadership Techniques Certification, University of London
- Data Science Specialization Certification, Coursera (Johns Hopkins University)
- B.Sc. in Computer Science, University of Edinburgh

Noah Williams is an ideal candidate for senior executive roles, combining technical expertise with strategic leadership.

- Dislplay the generated `interview_materials.md` file.

In [18]:
display(Markdown("./interview_materials.md"))

# Interview Questions and Talking Points for Noah Williams:

1. **Leadership and Team Management:**
   - Can you discuss a specific instance where you successfully managed remote and in-office engineering teams simultaneously? What challenges did you face, and how did you overcome them?
   - How do you ensure transparency and high performance across diverse engineering teams working in multiple time zones?

2. **Technical Expertise:**
   - With proficiency in multiple programming languages and frameworks, can you elaborate on a project where you utilized a diverse set of languages to achieve a specific goal?
   - How have your skills in data science and machine learning contributed to the success of your previous projects, particularly in deploying AI solutions?

3. **Strategic Thinking and Innovation:**
   - Describe a major tech initiative or startup where you led the team to achieve strategic project goals. What innovative approaches did you implement?
   - How do you foster a culture of collaboration and intelligence within your tech initiatives, driving growth and innovation in the industry?

4. **Project Experience:**
   - Share a challenging project experience where you successfully implemented AI and data science solutions. What were the key outcomes, and how did it impact the business?
   - Can you provide an example of a startup or tech initiative you led, highlighting the key strategies you employed to drive innovation and growth?

5. **Communication and Leadership Style:**
   - How do you approach problem-solving collaboratively within your teams, and what methods do you use to ensure clear and concise communication?
   - What leadership qualities do you believe are essential for driving growth and innovation in the tech industry, and how do you embody these qualities in your role?

These interview questions and talking points are designed to help you showcase your expertise, experience, and leadership qualities effectively during the interview process. Good luck!

# CONGRATULATIONS!!!

## Share your accomplishment!
- Once you finish watching all the videos, you will see the "In progress" image on the bottom left turn into "Accomplished".
- Click on "Accomplished" to view the course completion page with your name on it.
- Take a screenshot and share on LinkedIn, X (Twitter), or Facebook.  
- **Tag @Joāo (Joe) Moura, @crewAI, and @DeepLearning.AI, (and a few of your friends if you'd like them to try out the course)**
- **Joāo and DeepLearning.AI will "like"/reshare/comment on your post!**

## Get a completion badge that you can add to your LinkedIn profile!
- Go to [learn.crewai.com](https://learn.crewai.com).
- Upload your screenshot of your course completion page.
- You'll get a badge from CrewAI that you can share!

(Joāo will also talk about this in the last video of the course.)